# 05b_HDBSCAN_nivel2


In [1]:
import pandas as pd
import numpy as np
import time, joblib
from pathlib import Path
from datetime import datetime
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
DATA_MODELS = ROOT / 'data' / 'data_models_gustos'
INFORMES_BASE = ROOT / 'informes' / 'fase4_gustos' / '04_clustering' / 'nivel2'
DATA_MODELS.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_QC / 'features_tier2.parquet'
TIER_LEVEL = '2'

def reduce_high_card(df, max_unique=10):
    out = df.copy()
    for cat in out.select_dtypes(include=['object','category']).columns.tolist():
        if out[cat].nunique(dropna=True) > max_unique:
            top = out[cat].value_counts().head(max_unique).index.tolist()
            out[cat] = out[cat].where(out[cat].isin(top), 'OTHER')
    return out

def preprocess(df, nan_threshold_zero=0.30):
    df = df.drop(columns=['user_id']).copy()
    numeric = df.select_dtypes(include=['number']).columns.tolist()
    categorical = df.select_dtypes(include=['object','category']).columns.tolist()
    nan_pcts = df[numeric].isna().mean()
    num_low = nan_pcts[nan_pcts < nan_threshold_zero].index.tolist()
    num_high = nan_pcts[nan_pcts >= nan_threshold_zero].index.tolist()
    df = reduce_high_card(df, 10)
    pp = ColumnTransformer([
        ('num_low', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), num_low),
        ('num_high', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value=0)), ('sc', RobustScaler())]), num_high),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical),
    ])
    X = pp.fit_transform(df)
    return X, pp, pp.get_feature_names_out()

print(f'TIER {TIER_LEVEL}: cargando {MASTER_PATH.name}')
master = pd.read_parquet(MASTER_PATH)
user_ids = master['user_id'].values
print(f'  shape: {master.shape}')
X, preproc, names = preprocess(master)
joblib.dump(preproc, DATA_MODELS / f'preprocessor_tier{TIER_LEVEL}.joblib')
print(f'  X post-preproc: {X.shape}')
N = len(X)

import hdbscan
from sklearn.metrics import silhouette_score, davies_bouldin_score
REPORT = INFORMES_BASE / '05b_hdbscan'
REPORT.mkdir(parents=True, exist_ok=True)
MIN_SIZES = [50, 100, 250, 500]
SAMPLE_TRAIN = min(50_000, len(X))
SAMPLE_SIL = 20_000
results = []


TIER 2: cargando features_tier2.parquet
  shape: (21599, 17)
  X post-preproc: (21599, 16)


In [2]:
rng = np.random.RandomState(42)
if len(X) > SAMPLE_TRAIN:
    sample_idx = rng.choice(len(X), SAMPLE_TRAIN, replace=False)
    X_train = X[sample_idx]
else:
    sample_idx = np.arange(len(X))
    X_train = X

for ms in MIN_SIZES:
    t0 = time.time()
    cl = hdbscan.HDBSCAN(min_cluster_size=ms, metric='euclidean', core_dist_n_jobs=-1, prediction_data=True)
    cl.fit(X_train)
    labels_all, _ = hdbscan.approximate_predict(cl, X)
    n_clusters = int(len(set(labels_all)) - (1 if -1 in labels_all else 0))
    n_out = int((labels_all == -1).sum())
    non_out = labels_all != -1
    if non_out.sum() > 100 and len(set(labels_all[non_out])) > 1:
        sil_idx = rng.choice(np.where(non_out)[0], min(SAMPLE_SIL, int(non_out.sum())), replace=False)
        sil = float(silhouette_score(X[sil_idx], labels_all[sil_idx]))
        db = float(davies_bouldin_score(X[non_out], labels_all[non_out]))
    else:
        sil = db = float('nan')
    if n_clusters > 0:
        unique, counts = np.unique(labels_all[non_out], return_counts=True)
        max_pct = float(counts.max()/len(labels_all))
        min_pct = float(counts.min()/len(labels_all))
    else:
        max_pct = min_pct = float('nan')
    results.append({'algorithm':'HDBSCAN','tier':TIER_LEVEL,'min_cluster_size':ms,'n_clusters_actual':n_clusters,'n_outliers':n_out,'outlier_pct':n_out/len(labels_all),'silhouette':sil,'davies_bouldin':db,'max_cluster_pct':max_pct,'min_cluster_pct':min_pct,'elapsed_s':time.time()-t0})
    print(f'  min={ms}: clusters={n_clusters} out%={n_out/len(labels_all):.1%} sil={sil:.4f} max%={max_pct if pd.notna(max_pct) else "n/a"}')
    joblib.dump({'model':cl,'labels':labels_all,'user_ids':user_ids,'sample_idx':sample_idx}, DATA_MODELS / f'nivel{TIER_LEVEL}_hdbscan_min{ms}.joblib')

pd.DataFrame(results).to_csv(REPORT / 'metrics.csv', index=False)
print('OK metrics.csv')


  min=50: clusters=7 out%=43.6% sil=0.2524 max%=0.5173850641233391


  min=100: clusters=5 out%=50.7% sil=0.3015 max%=0.4571507940182416


  min=250: clusters=2 out%=61.1% sil=0.3350 max%=0.3766378073058938


  min=500: clusters=0 out%=100.0% sil=nan max%=n/a
OK metrics.csv


/Users/jezquerro/Documents/tfg/.venv/lib/python3.14/site-packages/hdbscan/prediction.py:391: UserWarning: Clusterer does not have any defined clusters, new data will be automatically predicted as noise.
  warn('Clusterer does not have any defined clusters, new data'
